# FFTW C2C P2PY MPI

In [41]:
#=-----------------------------------------------------------------------=#

In [21]:
%%writefile nc2cp.f90
subroutine fs(mr, ms, ss, tt1, tt2)
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer, intent(out) :: mr, ms
    double complex, intent(out) :: ss
    double precision, intent(out) :: tt1, tt2
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t0, t1, t2, t3

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t0)    ! <--- time measurement
    endif

    call fftw_mpi_init()    

    ! get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                 MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

    ! create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    if (mpirank == 0) then
      do k = 1, local_N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k + local_start))*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
      enddo
    endif

    ! compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    ! compute the checksum of processes
    s = sum(data)
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement
    endif
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    if (mpirank == 0) then
        call cpu_time(t3)    ! <--- time measurement
    endif
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    
    ! result
    mr = mpirank
    ms = mpisize
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement        
        ss = rs
        tt1 = t1-t0
        tt2 = t3-t2
        !write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        !write(*, "('  T: 'sf0.4' s')", advance="no") t1-t0
        !write(*, "('  Tc: 'sf0.4' s')", advance="no") t3-t2
        !write(*, "('  N: 'g0)") mpisize
    endif
    
    call mpi_finalize(mpierror)
end subroutine

Overwriting nc2cp.f90


In [22]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
f2py  -c nc2cp.f90  -m nc2cp  --f90exec=mpif90  \
-L$dir/lib  -lfftw3_mpi  -lfftw3  -lm  -I$dir/include  \
-DNPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION

running build
running config_cc
unifing config_cc, config, build_clib, build_ext, build commands --compiler options
running config_fc
unifing config_fc, config, build_clib, build_ext, build commands --fcompiler options
running build_src
build_src
building extension "nc2cp" sources
f2py options: []
f2py:> /tmp/tmpwydnn68g/src.linux-x86_64-3.8/nc2cpmodule.c
creating /tmp/tmpwydnn68g/src.linux-x86_64-3.8
Reading fortran codes...
	Reading file 'nc2cp.f90' (format:free)
{'before': '', 'this': 'use', 'after': ', intrinsic :: iso_c_binding '}
Line #2 in nc2cp.f90:"    use, intrinsic :: iso_c_binding "
	analyzeline: Could not crack the use statement.
Line #5 in nc2cp.f90:"    include 'fftw3-mpi.f03' "
	readfortrancode: could not find include file 'fftw3-mpi.f03' in . Ignoring.
Post-processing...
	Block: nc2cp
			Block: fs
In: :nc2cp:nc2cp.f90:fs
get_useparameters: no module mpi info used by fs
Post-processing (stage 2)...
Building modules...
	Building module "nc2cp"...
		Constructing wrapper f

nc2cp.f90:43.34:

            data(i, j, k) = cmplx(r, 0)
                                  1
nc2cp.f90:39.15:

      do k = 1, local_N
               1


In [23]:
%%writefile test01.py
import nc2cp
print(nc2cp.__doc__)

Overwriting test01.py


In [24]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
python test01.py

This module 'nc2cp' is auto-generated with f2py (version:2).
Functions:
  mr,ms,ss,tt1,tt2 = fs()
.


In [28]:
%%writefile test02.py
import nc2cp
mr, ms, ss, tt1, tt2 = nc2cp.fs()
if mr == 0 :
    print('S: {:.2f}  |  T: {:.4f}  |  Tc: {:.4f}  |  N: {:0g}'
        .format(ss, tt1, tt2, ms))

Overwriting test02.py


In [29]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpiexec -n 2 python test02.py

S: 125.00-0.00j  |  T: 4.6712  |  Tc: 0.0035  |  N: 2


In [30]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpiexec -n 9 python test02.py

S: 125.00+0.00j  |  T: 2.1481  |  Tc: 0.0560  |  N: 9


In [ ]:
! mv test02.py nc2cp_c.py

In [ ]:
cd /scratch${PWD#"/prj"}/
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu


In [43]:
! cp nc2cp* /scratch${PWD#"/prj"}

In [58]:
%%writefile nc2cp.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name nc2cp       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes
#   NTASKS: 1, 4, 9, 16, 36, 49, 64, 81
echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd /scratch${PWD#"/prj"}/
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
                                              
# Executable
EXEC="python nc2cp_c.py"

# Start
echo '$ srun --mpi=pmi2 -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun --mpi=pmi2 -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting nc2cp.srm


In [59]:
! sbatch --partition=cpu_dev --ntasks=1 nc2cp.srm

Submitted batch job 868502


In [60]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868502    cpu_dev   R  0:00     1   24


In [62]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [63]:
! cat /scratch${PWD#"/prj"}/slurm-868502.out

- Job ID: 868502
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun --mpi=pmi2 -n 1 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 7.1110  |  Tc: 0.0001  |  N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [64]:
! sbatch --partition=cpu_dev --ntasks=16 nc2cp.srm

Submitted batch job 868503


In [65]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868503    cpu_dev   R  0:02     1   24


In [66]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [67]:
! cat /scratch${PWD#"/prj"}/slurm-868503.out

- Job ID: 868503
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun --mpi=pmi2 -n 16 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 0.8961  |  Tc: 0.0000  |  N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 1 of (1, 4, 9, 16, 36, 49, 64, 81)

In [68]:
! sbatch --ntasks=1 nc2cp.srm
! sbatch --ntasks=1 nc2cp.srm
! sbatch --ntasks=1 nc2cp.srm

Submitted batch job 868504
Submitted batch job 868505
Submitted batch job 868506


In [69]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868504  cpu_small  PD  0:00     1    1
            868505  cpu_small  PD  0:00     1    1
            868506  cpu_small  PD  0:00     1    1


In [6]:
! cat /scratch${PWD#"/prj"}/slurm-868504.out
! cat /scratch${PWD#"/prj"}/slurm-868505.out
! cat /scratch${PWD#"/prj"}/slurm-868506.out

- Job ID: 868504
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 1 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 8.9111  |  Tc: 0.0001  |  N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868505
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 1 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 8.9269  |  Tc: 0.0001  |  N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868506
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 1 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 8.8605  |  Tc: 0.0001  |  N: 1
~~ end ~

### 4 of (1, 4, 9, 16, 36, 49, 64, 81)

In [70]:
! sbatch --ntasks=4 nc2cp.srm
! sbatch --ntasks=4 nc2cp.srm
! sbatch --ntasks=4 nc2cp.srm

Submitted batch job 868507
Submitted batch job 868508
Submitted batch job 868509


In [5]:
! cat /scratch${PWD#"/prj"}/slurm-868507.out
! cat /scratch${PWD#"/prj"}/slurm-868508.out
! cat /scratch${PWD#"/prj"}/slurm-868509.out

- Job ID: 868507
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 4 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 2.8153  |  Tc: 0.0014  |  N: 4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868508
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 4 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 2.8163  |  Tc: 0.0001  |  N: 4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868509
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 4 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 2.8150  |  Tc: 0.0001  |  N: 4
~~ end ~

### 9 of (1, 4, 9, 16, 36, 49, 64, 81)

In [71]:
! sbatch --ntasks=9 nc2cp.srm
! sbatch --ntasks=9 nc2cp.srm
! sbatch --ntasks=9 nc2cp.srm

Submitted batch job 868510
Submitted batch job 868511
Submitted batch job 868512


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-868510.out
! cat /scratch${PWD#"/prj"}/slurm-868511.out
! cat /scratch${PWD#"/prj"}/slurm-868512.out

- Job ID: 868510
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 9 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 3.3190  |  Tc: 0.0001  |  N: 9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868511
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 9 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 3.2972  |  Tc: 0.0327  |  N: 9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868512
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 9 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 3.3259  |  Tc: 0.0203  |  N: 9
~~ end ~

### 16 of (1, 4, 9, 16, 36, 49, 64, 81)

In [72]:
! sbatch --ntasks=16 nc2cp.srm
! sbatch --ntasks=16 nc2cp.srm
! sbatch --ntasks=16 nc2cp.srm

Submitted batch job 868513
Submitted batch job 868514
Submitted batch job 868515


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-868513.out
! cat /scratch${PWD#"/prj"}/slurm-868514.out
! cat /scratch${PWD#"/prj"}/slurm-868515.out

- Job ID: 868513
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 16 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 0.9617  |  Tc: 0.0000  |  N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868514
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 16 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 0.9587  |  Tc: 0.0059  |  N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868515
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun --mpi=pmi2 -n 16 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 0.9487  |  Tc: 0.0028  |  N: 16

### 36 of (1, 4, 9, 16, 36, 49, 64, 81)

In [73]:
! sbatch --ntasks=36 nc2cp.srm
! sbatch --ntasks=36 nc2cp.srm
! sbatch --ntasks=36 nc2cp.srm

Submitted batch job 868516
Submitted batch job 868517
Submitted batch job 868518


In [1]:
! cat /scratch${PWD#"/prj"}/slurm-868516.out
! cat /scratch${PWD#"/prj"}/slurm-868517.out
! cat /scratch${PWD#"/prj"}/slurm-868518.out

- Job ID: 868516
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
$ srun --mpi=pmi2 -n 36 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 1.2719  |  Tc: 0.0035  |  N: 36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868517
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
$ srun --mpi=pmi2 -n 36 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 1.2562  |  Tc: 0.0029  |  N: 36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868518
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
$ srun --mpi=pmi2 -n 36 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |

### 49 of (1, 4, 9, 16, 36, 49, 64, 81)

In [74]:
! sbatch --ntasks=49 nc2cp.srm
! sbatch --ntasks=49 nc2cp.srm
! sbatch --ntasks=49 nc2cp.srm

Submitted batch job 868519
Submitted batch job 868520
Submitted batch job 868521


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-868519.out
! cat /scratch${PWD#"/prj"}/slurm-868520.out
! cat /scratch${PWD#"/prj"}/slurm-868521.out

- Job ID: 868519
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
$ srun --mpi=pmi2 -n 49 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 1.1818  |  Tc: 0.0037  |  N: 49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868520
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1353
$ srun --mpi=pmi2 -n 49 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 1.5511  |  Tc: 0.0034  |  N: 49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868521
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
$ srun --mpi=pmi2 -n 49 python nc2cp_c.py
-- output ------------

### 64 of (1, 4, 9, 16, 36, 49, 64, 81)

In [75]:
! sbatch --ntasks=64 nc2cp.srm
! sbatch --ntasks=64 nc2cp.srm
! sbatch --ntasks=64 nc2cp.srm

Submitted batch job 868522
Submitted batch job 868523
Submitted batch job 868524


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-868522.out
! cat /scratch${PWD#"/prj"}/slurm-868523.out
! cat /scratch${PWD#"/prj"}/slurm-868524.out

- Job ID: 868522
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1353
$ srun --mpi=pmi2 -n 64 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 1.1820  |  Tc: 0.0068  |  N: 64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868523
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
$ srun --mpi=pmi2 -n 64 python nc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j  |  T: 1.4662  |  Tc: 0.0167  |  N: 64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868524
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1353
$ srun --mpi=pmi2 -n 64 python nc2cp_c.py
-- output ------------

### 81 of (1, 4, 9, 16, 36, 49, 64, 81)

In [76]:
! sbatch --ntasks=81 nc2cp.srm
! sbatch --ntasks=81 nc2cp.srm
! sbatch --ntasks=81 nc2cp.srm

Submitted batch job 868525
Submitted batch job 868526
Submitted batch job 868527


In [77]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868509  cpu_small  PD  0:00     1    4
            868507  cpu_small  PD  0:00     1    4
            868508  cpu_small  PD  0:00     1    4
            868505  cpu_small  PD  0:00     1    1
            868506  cpu_small  PD  0:00     1    1
            868504  cpu_small  PD  0:00     1    1
            868519  cpu_small  PD  0:00     3   49
            868520  cpu_small  PD  0:00     3   49
            868521  cpu_small  PD  0:00     3   49
            868518  cpu_small  PD  0:00     2   36
            868517  cpu_small  PD  0:00     2   36
            868516  cpu_small  PD  0:00     2   36
            868515  cpu_small  PD  0:00     1   16
            868513  cpu_small  PD  0:00     1   16
            868514  cpu_small  PD  0:00     1   16
            868511  cpu_small  PD  0:00     1    9
            868512  cpu_small  PD  0:00     1    9
            868510  cpu_small  PD  0:00     1    9
            868522  cpu_small  

In [78]:
! squeue -u $(whoami) -h -t pending,running -r | wc -l

48


In [1]:
! squeue -n nc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-868525.out
! cat /scratch${PWD#"/prj"}/slurm-868526.out
! cat /scratch${PWD#"/prj"}/slurm-868527.out

- Job ID: 868525
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
$ srun --mpi=pmi2 -n 81 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 1.3464  |  Tc: 0.1016  |  N: 81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868526
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1337 sdumont1338
$ srun --mpi=pmi2 -n 81 python nc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j  |  T: 1.4284  |  Tc: 0.0026  |  N: 81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868527
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
$ srun --mpi=pmi2 -n 81 pyth